<a href="https://colab.research.google.com/github/pmcray/worldmaker/blob/claude/status-colab-notebooks-w4tnm6/traveller_world_generator.ipynb" target="_parent"><img
src="https://colab.research.google.com/assets/colab-badge.svg"
alt="Open In Colab"/></a>

# Traveller Sector Generator — Foreven Sector (1105) & Custom Universe Builder

Generates the **Foreven Sector** (*Iakr* in Zhodani) or a custom sector and renders it in the classic Traveller style using the `worldmaker` package.

### Default Configuration: Strict 1105 Canon
Out of the box, this notebook defaults to the **absolute minimum: Strict 1105 Canon** (FFE 2008 *Foreven Sector Reserve* + Classic Traveller primary citations):
- **358 canonical star positions** and **142 Zhodani Consulate hexes** (`ZhIN`, *Preqlianz of Iadr Nsobl*).
- **Only the 4 canonical FFE subsector names**: `D: Massina`, `H: Fessor`, `L: Reidain`, `P: Urnian`.
- **The 3 canonical statted worlds**: `Zdovesil (1212)`, `Alenzar (3229)` (with secondary planet *Chamax* from *Double Adventure 5*), and `Raschev (3230)`.
- **The 3 Classic Traveller named unstatted systems**: `Tlebria (1618)` (*Book 7*), `Purfyr (2211)` (*JTAS #16*), and `Cocta (2415)` (*JTAS #2*).
- **Canonical Minor Races & Aliens**: Two *Alien Module 5* Droyne worlds (`0513` in Pieplow, `3003` in Massina) and `Parthinia (3018)` (*Alien Realms* "The Deep Metal", homeworld of the *Issugur* minor human race).
- **Canonical Corporations**: `Iadria Vlovl`, `Priantqlovr Drafr`, and `InStarSpec`.

Every deuterocanonical, *Sector Construction Guide* (SCG 2022), third-party, settlement wave, and custom *Referee's Preserve* faction (`Heptamorphs`, `Uplifted Terran Cats`, `Halfling Shires`, `Christen Erith`) is **toggleable below**. Furthermore, **every run always identifies the most Earth-like worlds and Solar System-like systems**.

In [ ]:
# --- Colab bootstrap -------------------------------------------------
# On Google Colab this fetches worldmaker from GitHub, so there is nothing
# to install by hand. Run from a clone of the repository it finds the local
# package instead and does nothing. Either way it is safe to re-run.
WORLDMAKER_REF = "claude/status-colab-notebooks-w4tnm6"     # branch, tag or commit to fetch on Colab

import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/pmcray/worldmaker.git"
CLONE = "/content/worldmaker"

if IN_COLAB:
    installed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         f"git+{REPO}@{WORLDMAKER_REF}"]).returncode == 0

    if not installed:
        # A ref from before the package carried its packaging metadata:
        # clone it and import it where it stands.
        print(f"pip install of {WORLDMAKER_REF} failed; cloning instead")
        if not os.path.isdir(CLONE):
            subprocess.run(["git", "clone", "--depth", "1",
                            "--branch", WORLDMAKER_REF, REPO, CLONE],
                           check=True)
        sys.path.insert(0, CLONE)

    # Canon fetched from travellermap.com is cached beside the notebook
    # rather than inside site-packages.
    os.environ.setdefault("WORLDMAKER_CANON_CACHE", "/content/canon_cache")

try:
    import worldmaker as wm
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "worldmaker not found. Start Jupyter from the root of a clone of "
        "the repository, or install it with\n"
        f"    pip install git+{REPO}") from None

print("worldmaker", getattr(wm, "__version__", "(unversioned)"),
      "ready" + (" on Colab" if IN_COLAB else ""))

In [ ]:
import random
import worldmaker as wm
from IPython.display import SVG, Markdown, display

SEED = 1105

# Choose a preset ("strict_1105", "deuterocanon_1105", "scg_worked_1105",
# "community_synthesis_1105", "referee_preserve_1107") or customize individual
# scenario toggles directly on ForevenScenario:
scenario = wm.ForevenScenario(
    current_year=1105,
    canonical_positions_358=True,
    zhodani_consulate_border=True,
    subsector_naming_mode="ffe_4",        # "ffe_4" | "imperial_lines_16" | "zhodani_base_16" | "custom_16"

    # Tier 1: Strict 1105 Canon (ON by default)
    strict_1105_worlds=True,              # Zdovesil (1212), Alenzar (3229)+Chamax, Raschev (3230)
    classic_named_systems=True,           # Tlebria (1618), Purfyr (2211), Cocta (2415)
    include_cocta_2415=False,
    am5_droyne_worlds=True,               # Droyne at 0513 & 3003 (Alien Module 5)
    parthinia_issugur=True,               # Parthinia (3018) Issugur minor human race
    canonical_corporations=True,          # Iadria Vlovl, Priantqlovr Drafr, InStarSpec

    # Tier 2: Deuterocanonical Layers (OFF by default)
    imperial_lines_anchors=False,         # Hollis (2523), Avalar (1636), Gatina, Sturray, Ranther
    hollis_babylon5_mode=False,           # Scout-only diplomatic station mode for Hollis
    rimward_polity_mode="none",           # "none" | "weltbund_1105" | "avalar_il1" | "avalar_scg_27"
    mnemosyne_principality=False,         # Pocket empire in Subsectors M/N ("Mn")

    # Tier 3: Sector Construction Guide (SCG 2022) Layers (OFF by default)
    scg_tlesho_union=False,               # 4-world Tlinzha octopod polity ("TlU")
    scg_native_sophonts=False,            # 7 SCG native sophont homeworlds
    scg_ancients_ruins=False,             # 4 SCG Ancients ruins sites
    scg_scattered_settlements=False,      # 10 SCG scattered settlements
    scg_barren_anomaly=False,             # 14 SCG barren systems in Subsector J & C
    scg_strategic_bases=False,            # Zhodani waypoints, Imperial Droyne Cage, TL ceilings

    # Tier 4: Third-Party & Community Layers (OFF by default)
    zhodani_base_reidain=False,           # Reidain (2925) TL13 waterworld + Kamrathi League ("Km")
    jbe_foreven_worlds=False,             # JBE Cornathian Empire ("Co") + Warpride (2136)
    mongoose_1e_reidain_salient=False,    # 2008 Mongoose 1e Zhodani salient into Reidain

    # Tier 5: Settlement Wave Modeling (SCG pp. 22-27)
    enable_settlement_waves=True,
    wave_zhodani_thick=True,              # Coreward Zhodani thick wave (-5000 to -1000)
    wave_imperial_thin=True,              # Trailing Imperial thin wave (50 to 589)
    wave_zhodani_thin=True,               # Post-FFW II Zhodani thin wave (600 to 700)
    wave_rimward_hub=False,               # Radial wave from Avalar 1636
    wave_psionic_refugees=False,          # Post-800 Psionic Suppressions wave

    # Tier 6: Procedural & Custom Referee's Preserve Factions (OFF by default)
    allow_random_native_sophonts=False,
    allow_procedural_polities=False,
    custom_heptamorphs=False,             # Heptarchy of Septima ("He")
    custom_uplifted_cats=False,           # Chatelaine of Bastet ("Fe")
    custom_halfling_shires=False,         # Periannath Shires ("Hl")
    christen_erith=False,                 # Rename #1 Earth analogue to "Erith"
)

sector = wm.generate_foreven_1105(seed=SEED, scenario=scenario)
print(f"{len(sector.systems)} systems in {sector.width}x{sector.height} hexes")
print(f"{len(sector.polities)} polities, "
      f"{sum(1 for r in sector.routes if r[2] == 'xboat')} Xboat links, "
      f"{len(sector.settlement_waves)} active settlement waves")

## Polities & Canonical Corporations

In [ ]:
for p in sector.polities:
    print(f"{p.allegiance_code:4s}  {p.name:34s} capital {p.capital_hex or '-':4s}  "
          f"{len(p.controlled_systems):3d} worlds  DI {p.defense_index:2d}  "
          f"{p.polity_type}")

unaligned = sum(1 for s in sector.systems.values() if s.allegiance in ("Na", "XXXX", ""))
print(f"\n{unaligned} non-aligned / unclaimed reserve systems")

if sector.corporations:
    print("\nCanonical Corporations:")
    for c in sector.corporations:
        print(f"  • {c['name']:18s} HQ: {c['hq_hex']} {c['hq_world']:28s} ({c['allegiance']}) — {c['role']}")

## Subsectors

In [ ]:
for idx in range(16):
    letter = chr(65 + idx)
    name = sector.subsector_names.get(letter, "(Unnamed in Strict 1105 Canon)")
    col0, row0 = (idx % 4) * 8, (idx // 4) * 10
    count = sum(
        1 for h in sector.systems
        if col0 < int(h[:2]) <= col0 + 8 and row0 < int(h[2:]) <= row0 + 10)
    print(f"{letter}  {name:32s} {count:3d} systems")

## Settlement Waves & Sector History (SCG pp. 22–27)

Models how directional and point-radial settlement waves propagate across Foreven,
computing local arrival years, centuries of settlement maturity, SCG p. 22 Population DMs,
and multi-wave historical strata on systems crossed by successive waves.

In [ ]:
waves_df = wm.create_settlement_waves_dataframe(sector)
display(waves_df)

strata_df = wm.create_settlement_strata_dataframe(sector)
display(Markdown("**Sample Multi-Wave Systems (Settled by 2+ Historical Waves):**"))
display(strata_df[strata_df["Wave Count"] >= 2].head(12))

In [ ]:
display(SVG(wm.generate_settlement_wave_svg(sector)))

## Most Earth-like Worlds & Solar System-like Systems

Automatically evaluated across every run (independent of custom faction toggles):
- **Top Earth-like Worlds (`sector.earth_analogues`)**: Coupled Earth Similarity Index (physical ESI + host star class, orbital habitable zone, stellar age, and metallicity `[Fe/H]`).
- **Top Solar System Architectural Analogues (`sector.solar_system_analogues`)**: Single Solar-like star, inner rocky terrestrials, zero Hot Jupiters, outer gas giants, and planetoid belts.
- **Top Solar Analogue Stars (`sector.solar_analogues`)**: Ranked against Sol (`G2 V`, 4.6 Gyr, `[Fe/H] = 0.00`).

In [ ]:
display(Markdown(wm.describe_sector_analogues(sector)))
analogues = wm.create_analogues_dataframes(sector)

display(Markdown("### Top 10 Earth-like Worlds"))
display(analogues["earth_analogues"])

display(Markdown("### Top 10 Solar System Architectural Analogues"))
display(analogues["solar_system_analogues"])

display(Markdown("### Top 10 Solar Analogue Stars"))
display(analogues["solar_analogues"])

## Sector map

The full foldout, in the classic black-ink style.

In [ ]:
display(SVG(wm.generate_sector_svg(sector)))

## Subsector map

A single subsector at Supplement-page scale.

In [ ]:
display(SVG(wm.generate_subsector_svg(
    sector, sector.subsector_names.get("L", "Subsector L: Reidain"),
    origin_col=25, origin_row=21)))

### Any other subsector

Pass the origin hex of the subsector window: subsector F starts at 0911 (containing Zdovesil at 1212 and Tlebria at 1618).

In [ ]:
display(SVG(wm.generate_subsector_svg(
    sector, sector.subsector_names.get("F", "Subsector F"),
    origin_col=9, origin_row=11)))

## Building a sector from a universe

The *Sector Construction Guide* opens by asking a handful of questions
before any dice are rolled (pp.3-8): what the theme is, how densely the
stars are packed, how advanced the setting is, whether it is growing or
falling, and whether any anomaly bends the rules over part of the map.
`create_universe()` takes those answers and drives generation from them.

In [ ]:
universe = wm.create_universe("Foreven Reach", preset="frontier")

# A "Bermuda Triangle" where colonies fail, and a rift that thins the stars
universe.add_anomaly("barren", "The Quiet Zone", centre="1124", radius=2)
universe.add_anomaly("rift", "The Gap", centre="0505", radius=2)

wm.build_default_timeline(universe)
print(universe.describe())

In [ ]:
random.seed(1105)
themed = wm.generate_universe_sector(universe, "Foreven Reach")
print(f"{len(themed.systems)} systems, {len(themed.polities)} polities")

# The rift is empty, and the Quiet Zone's worlds stand abandoned
rift = universe.anomalies[1].hexes
print(f"systems inside the rift: {sum(1 for h in rift if h in themed.systems)}")

for hex_coord in universe.anomalies[0].hexes:
    system = themed.systems.get(hex_coord)
    if system and system.mainworld:
        print(f"  {hex_coord}  {system.mainworld.uwp}  "
              f"{' '.join(system.mainworld.trade_codes)}")

## Sector data file

Traveller5 Second Survey data, in the column-delimited format
travellermap.com defines. The separator line fixes each field's width, so
every value sits in its own column: Hex, Name, UWP, Remarks, then the
importance, economic and cultural extensions, nobility, bases, travel
zone, PBG, world count, allegiance and stellar data.

In [ ]:
sec_data = wm.export_sector_sec_file(sector)
print("\n".join(sec_data.splitlines()[:25]))
print("...")

In [ ]:
# Write the maps and data to disk
with open("sector_map.svg", "w") as f:
    f.write(wm.generate_sector_svg(sector))
with open("sector_waves_map.svg", "w") as f:
    f.write(wm.generate_settlement_wave_svg(sector))
with open("sector.sec", "w") as f:
    f.write(sec_data)
print("written: sector_map.svg, sector_waves_map.svg, sector.sec")

In [ ]:
# On Colab, take the map and the sector data away with you before the
# runtime is recycled.
import sys
from pathlib import Path

written = ["sector_map.svg", "sector.sec"]

if "google.colab" in sys.modules:
    from google.colab import files
    for name in written:
        files.download(name)
else:
    for name in written:
        print(Path(name).resolve())